# Session 1 — Register to Unity Catalog & Deploy Serving Endpoint

**Goal:** Take the best model from our MLflow experiment and:
1. Register it in the Unity Catalog Model Registry
2. Assign it the `champion` alias
3. Deploy it as a real-time serving endpoint
4. Query it with a sample customer record

In [0]:
%run ../utils/config

In [0]:
%pip install ../bundle/wheels/ databricks-feature-engineering --quiet

## Provide your Best Run ID
1. If you haven't already done so, paste the value of your **best_run_id** in the notebook parameter at the top of the page.

In [0]:

dbutils.widgets.text("best_run_id", "", "Best Run ID (from 04_mlflow_experiment)")

best_run_id = dbutils.widgets.get("best_run_id")

if not best_run_id:
    # Try to get from task values (when running as a job)
    try:
        best_run_id = dbutils.jobs.taskValues.get(taskKey="train_models", key="best_run_id")
    except Exception:
        raise ValueError("Please enter the best_run_id from 04_mlflow_experiment.ipynb")

## MLflow model registry
MLflow can utilize the registry of your choice.  By default in Databricks, MLflow will register models in the workspace files space.  In the following cell, we instruct MLflow to register the model with Unity Catalog so that we can utilize Unity Catalog's full governance for the model.

In [0]:
import mlflow
from mlflow import MlflowClient

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Instantiate a MLflow client to be used later
client = MlflowClient()

model_name = f"{catalog}.{schema}.churn_classifier"
print(f"Model name  : {model_name}")
print(f"Best run ID : {best_run_id}")

## Step 1: Register the Model in Unity Catalog

Registering a model in Unity Catalog:
- Creates a versioned, governed model artifact
- Links the model back to the MLflow run (automatic lineage!)
- Enables you to assign human-readable aliases like `@champion`

In [ ]:
# Register the pyfunc / feature-store artifact as @champion.
# This version carries feature-lookup metadata so fe.score_batch() can retrieve
# feature values automatically from the offline feature table by customerID.
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=model_name,
)

print(f"✓ Registered: {registered.name}")
print(f"  Version   : {registered.version}")
print(f"  Status    : {registered.status}")

In [0]:
# Assign the 'champion' alias — this is what Model Serving and batch jobs will reference
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered.version,
)
print(f"✓ Alias 'champion' set on version {registered.version}")
print(f"\nModel URI with alias: models:/{model_name}@champion")

In [ ]:
# Also register the raw sklearn pipeline for the serving endpoint.
# The pyfunc artifact (@champion) needs an Online Table for real-time feature lookup —
# an infrastructure component we don't set up in this workshop.
# The raw sklearn pipeline receives full features as input and deploys without it.
serving_registered = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model_pipeline",
    name=model_name,
)

client.set_registered_model_alias(
    name=model_name,
    alias="serving",
    version=serving_registered.version,
)

print(f"✓ Alias '@serving' set on version {serving_registered.version}  (raw sklearn pipeline)")
print()
print(f"  @champion  → v{registered.version}    pyfunc / feature-store  →  fe.score_batch()")
print(f"  @serving   → v{serving_registered.version}    raw sklearn pipeline    →  serving endpoint")

## Step 2: Create a Model Serving Endpoint (Illustrative)

> **Batch scoring via `fe.score_batch()` (Step 3) is the primary production pattern for a churn model** — predictions are written to a Delta table and consumed by CRM/marketing systems on a nightly or weekly schedule.
>
> A **serving endpoint** is useful when you need an on-demand score: for example, a customer-support tool that needs a risk score the moment a customer calls. We show the full setup here so you can see how it's done.

Databricks Model Serving creates a managed REST endpoint that:
- Auto-scales from zero (no cost when idle)
- Handles versioning and traffic routing
- Logs every request/response to an **AI Gateway inference table**
- **Rate-limits requests** via AI Gateway (60 calls/min per user)

We use the `@serving` version (raw sklearn pipeline) — it accepts all feature values as input and works without an Online Table.

We start creation **asynchronously** and continue immediately to Step 3. The endpoint will be ready in 5-10 minutes — you can run the REST test cell at the end of this notebook once it shows **Ready** in the Serving UI.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    EndpointCoreConfigInput,
    ServedModelInput,
)

w = WorkspaceClient()
endpoint_name = f"churn_{safe_username}"

# create() fires asynchronously — returns immediately while the endpoint spins up.
# Use create_and_wait() if you want to block until Ready (adds 5-10 minutes).
w.serving_endpoints.create(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        name=endpoint_name,
        served_models=[
            ServedModelInput(
                model_name=model_name,
                model_version=serving_registered.version,
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix="churn",
            enabled=True,
        ),
        rate_limits=[
            AiGatewayRateLimit(
                calls=60,
                key=AiGatewayRateLimitKey.USER,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            )
        ],
    ),
)
print(f"✓ Endpoint creation started: {endpoint_name}")
print(f"  Model version : {serving_registered.version}  (@serving — raw sklearn pipeline)")
print(f"  Inference table will appear at: {catalog}.{schema}.churn_payload")
print()
print("  → Continue to Step 3. Check Serving > {endpoint_name} for status.")

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    EndpointCoreConfigInput,
    ServedModelInput,
)

w = WorkspaceClient()
endpoint_name = f"churn_{safe_username}"

print(f"Creating endpoint : {endpoint_name}")
print(f"Model version     : {registered.version}")
print("This will take 5-10 minutes...")

w.serving_endpoints.create_and_wait(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        name=endpoint_name,
        served_models=[
            ServedModelInput(
                model_name=model_name,
                model_version=registered.version,
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix="churn",
            enabled=True,
        ),
        rate_limits=[
            AiGatewayRateLimit(
                calls=60,
                key=AiGatewayRateLimitKey.USER,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            )
        ],
    ),
)
print(f"✓ Endpoint '{endpoint_name}' is Ready!")
print(f"  Inference table: {catalog}.{schema}.churn_payload")
print(f"  Rate limit     : 60 requests/min per user")

### Monitor Endpoint Creation (Background)

1. Click **Serving** in the left sidebar
2. Find <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_{safe_username}</span> — status will show **Creating**
3. **Continue with Step 3 below** — batch inference and the LLM explanation do not require the endpoint to be ready
4. Come back to the REST test cell (near the bottom) once status shows **Ready**

Observe the endpoint configuration when it's ready:
- Served model: `@serving` alias (raw sklearn pipeline)
- Scale-to-zero enabled
- AI Gateway: inference table + rate limits configured

### Navigate to Model Serving

1. Click **Serving** in the left sidebar
2. Find <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_{safe_username}</span>
3. When the status shows **Ready**, proceed to the next cell

Observe the endpoint configuration:
- Served model version
- Scale-to-zero setting
- Use → Query (**try it!**)

## Step 3: Test


import mlflow
import time
import numpy as np
import pandas as pd

start = time.time()
model = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")
print(f"Model loaded in {time.time() - start:.1f}s")

sample_input = pd.DataFrame([{
    "customerID":       "TEST_001",
    "gender":           "Female",
    "SeniorCitizen":    np.int32(0),
    "Partner":          "No",
    "Dependents":       "No",
    "tenure":           np.int32(1),
    "PhoneService":     "Yes",
    "MultipleLines":    "No",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "No",
    "OnlineBackup":     "No",
    "DeviceProtection": "No",
    "TechSupport":      "No",
    "StreamingTV":      "Yes",
    "StreamingMovies":  "Yes",
    "Contract":         "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Electronic check",
    "MonthlyCharges":   85.7,
    "TotalCharges":     85.7,
},
{
    "customerID":       "TEST_002",
    "gender":           "Male",
    "SeniorCitizen":    np.int32(0),
    "Partner":          "Yes",
    "Dependents":       "Yes",
    "tenure":           np.int32(72),
    "PhoneService":     "Yes",
    "MultipleLines":    "Yes",
    "InternetService":  "Fiber optic",
    "OnlineSecurity":   "Yes",
    "OnlineBackup":     "Yes",
    "DeviceProtection": "Yes",
    "TechSupport":      "Yes",
    "StreamingTV":      "Yes",
    "StreamingMovies":  "Yes",
    "Contract":         "Two year",
    "PaperlessBilling": "Yes",
    "PaymentMethod":    "Credit card (automatic)",
    "MonthlyCharges":   114.05,
    "TotalCharges":     8468.2,
}])

local_predictions = model.predict(sample_input)
# Store TEST_001's prediction — used by the LLM explanation below
local_prediction = local_predictions[0]

display(pd.DataFrame({
    "customerID": sample_input["customerID"],
    "prediction": local_predictions,
}))

In [ ]:
#### Feature Store Batch Inference — The Primary Production Pattern

For a churn model, **batch scoring is the right architecture**. A nightly or weekly job:
1. Pulls customer IDs from your data warehouse
2. Looks up current feature values from the Feature Store (offline Delta table)
3. Scores all customers in a single Spark job
4. Writes predictions to a Delta table
5. CRM / marketing systems read predictions from that table

`fe.score_batch()` implements this in one call — it handles the feature lookup automatically.

The two UC aliases map directly to the two deployment patterns:

| Alias | Artifact | Pattern |
|---|---|---|
| `@champion` | `model/` (pyfunc + feature-store) | `fe.score_batch()` — key-based lookup from offline Delta table |
| `@serving` | `model_pipeline/` (raw sklearn) | Serving endpoint — receives full features via REST |

from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Only the lookup key is needed — feature values are fetched automatically
feature_table = f"{catalog}.{schema}.churn_features"
customer_ids_df = spark.table(feature_table).limit(5).select("customerID")

predictions_df = fe.score_batch(
    model_uri=f"models:/{model_name}@champion",
    df=customer_ids_df,
)

display(predictions_df.select("customerID", "prediction"))

In [ ]:
#### REST Interface

> **Run this once the endpoint is ready.** Check **Serving** > `churn_{safe_username}` — status must show **Ready**. If it's still **Creating**, skip to the LLM explanation below — it uses the local prediction from the pyfunc test above and doesn't need the endpoint.

This shows how any application can call the model over HTTP using the `@serving` version (raw sklearn pipeline). The request body includes all feature values; the endpoint scores and returns predictions immediately.

#### REST Interface
Use REST to make an inference

In [ ]:
## Bonus: LLM-Powered Churn Explanation

The churn model returns a prediction, but **why** did this customer churn?

We route a follow-up call through the **AI Gateway** to a Foundation Model API endpoint to generate a human-readable explanation. This uses the local prediction from the pyfunc test above — no need to wait for the serving endpoint.

A common endpoint, <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">foundation_model_with_gateway</span>, has been created for you with the following configuration:
 - Model <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">`system.ai.llama_v3_3_70b_instruct`
 - Provisioned Throughput with scale-to-zero
 - Input and Output guardrails
 - Usage tracking enabled
 - Inference table enabled
 - Rate limited to 60 QPM per user

This demonstrates:
- **AI Gateway** as a unified proxy for both custom ML endpoints and LLMs
- **Foundation Model API** — any frontier model running within the Databricks platform security boundary
- How GenAI augments traditional ML in production workflows

import requests, json

# Auth — reuse the WorkspaceClient created during endpoint setup
workspace_url = w.config.host
headers = {"Content-Type": "application/json"}
headers.update(w.config.authenticate())

fm_endpoint_name = "foundation_model_with_gateway"
fm_url = f"{workspace_url}/serving-endpoints/{fm_endpoint_name}/invocations"

# Use the REST prediction if cell-21 was run; otherwise fall back to the local
# pyfunc prediction from above (works whether or not the endpoint is ready yet).
try:
    prediction = result.get("predictions", [local_prediction])[0]
    customer = sample_customer["dataframe_records"][0]
except NameError:
    prediction = local_prediction
    customer = sample_input.iloc[0].to_dict()

customer_lines = "\n".join(
    f"  {k}: {v}" for k, v in customer.items() if k != "customerID"
)

prompt = f"""A customer churn prediction model predicted: {prediction}

Customer profile:
{customer_lines}

In 2-3 sentences, explain the most likely reasons for this prediction based on
the customer profile, and suggest one specific retention action.
"""

llm_response = requests.post(
    fm_url,
    headers=headers,
    json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200,
    },
    timeout=180,
)

llm_result = llm_response.json()

if llm_response.status_code != 200:
    print(f"\u26a0\ufe0f  LLM request failed \u2014 HTTP {llm_response.status_code}")
    print(f"   Error code: {llm_result.get('error_code', 'N/A')}")

    detail = llm_result.get("detail", llm_result.get("message", "No details available"))

    if isinstance(detail, str):
        try:
            detail = json.loads(detail)
        except (json.JSONDecodeError, ValueError):
            pass

    if isinstance(detail, dict):
        reason = detail.get("finishReason", "unknown")
        print(f"   Finish reason: {reason}")

        for guard in detail.get("input_guardrail", []):
            if guard.get("flagged"):
                flagged_cats = [k for k, v in guard.get("categories", {}).items() if v]
                print(f"   Flagged categories: {', '.join(flagged_cats)}")

        for guard in detail.get("output_guardrail", []):
            if guard.get("flagged"):
                flagged_cats = [k for k, v in guard.get("categories", {}).items() if v]
                print(f"   Flagged categories (output): {', '.join(flagged_cats)}")

        print(f"\n   Full detail:\n{json.dumps(detail, indent=2)}")

    else:
        print(f"\nMessage:\n{detail}")

else:
    explanation = llm_result["choices"][0]["message"]["content"]
    print(f"Churn prediction : {prediction}")
    print(f"FM Gateway       : {fm_endpoint_name}")
    print(f"\nLLM explanation:\n{explanation}")

In [0]:
# Query the LLM through the FM Gateway endpoint created in the previous cell
fm_endpoint_name = "foundation_model_with_gateway"
fm_url = f"{workspace_url}/serving-endpoints/{fm_endpoint_name}/invocations"

# Extract the prediction from the churn endpoint response
prediction = result.get("predictions", ["unknown"])[0]

# Build a concise customer summary for the LLM
customer = sample_customer["dataframe_records"][0]
customer_lines = "\n".join(
    f"  {k}: {v}" for k, v in customer.items() if k != "customerID"
)

prompt = f"""A customer churn prediction model predicted: {prediction}

Customer profile:
{customer_lines}

In 2-3 sentences, explain the most likely reasons for this prediction based on
the customer profile, and suggest one specific retention action.
"""

# Call the model
llm_response = requests.post(
    fm_url,
    headers=headers,
    json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200,
    },
    timeout=180,
)

llm_result = llm_response.json()

# Check for errors
if llm_response.status_code != 200:
    print(f"\u26a0\ufe0f  LLM request failed \u2014 HTTP {llm_response.status_code}")
    print(f"   Error code: {llm_result.get('error_code', 'N/A')}")

    detail = llm_result.get("detail", llm_result.get("message", "No details available"))

    # If detail is a JSON string, parse it into a dict first
    if isinstance(detail, str):
        try:
            detail = json.loads(detail)
        except (json.JSONDecodeError, ValueError):
            pass

    if isinstance(detail, dict):
        reason = detail.get("finishReason", "unknown")
        print(f"   Finish reason: {reason}")

        # Show flagged categories if this was a guardrail violation
        for guard in detail.get("input_guardrail", []):
            if guard.get("flagged"):
                flagged_cats = [k for k, v in guard.get("categories", {}).items() if v]
                print(f"   Flagged categories: {', '.join(flagged_cats)}")

        for guard in detail.get("output_guardrail", []):
            if guard.get("flagged"):
                flagged_cats = [k for k, v in guard.get("categories", {}).items() if v]
                print(f"   Flagged categories (output): {', '.join(flagged_cats)}")

        print(f"\n   Full detail:\n{json.dumps(detail, indent=2)}")

    else:
        print(f"\nMessage:\n{detail}")

else:
    explanation = llm_result["choices"][0]["message"]["content"]
    print(f"Churn prediction : {prediction}")
    print(f"FM Gateway       : {fm_endpoint_name}")
    print(f"\nLLM explanation:\n{explanation}")

### Guardrails
1. In the previous cell, insert a new line in the prompt (line 21) and add something like 
   -  **"The customer's SSN is 123-45-6789"**
1. Rerun the cell.
1. Observe how the guardrail detects outgoing PII.

### AI Gateway — What We Just Enabled

The endpoint above uses the AI Gateway embedded directly in Model Serving. All governance features are available today via API:

| Feature | Available |
|---|---|
| Inference table logging | ✅ Yes |
| Rate limiting | ✅ Yes |
| PII detection & blocking | ✅ Yes |
| Toxicity / safety guardrails | ✅ Yes |

  
  
Both endpoints (`churn_{username}` and `foundation_model_with_gateway`) use the same gateway configuration pattern — giving you a consistent governance layer across custom ML models and Foundation Model API calls.

## Summary

✅ **Session 1 complete!** You have:

1. Explored the Telco dataset and understood its characteristics
2. Refactored a messy notebook into a modular, testable package
3. Trained 3 model types and compared them in MLflow
4. Registered the best model in Unity Catalog with a `@champion` alias
5. Deployed a rate-limited serving endpoint with AI Gateway
6. Made a live prediction via REST API
7. Generated an LLM explanation using Foundation Model API


**Before Session 2**, review [06_production_checklist.ipynb]($./06_production_checklist) to see what's still missing.